In [0]:
%pip install pysus

In [0]:
%pip install pyreaddbc dbfread
dbutils.library.restartPython()

In [0]:
import urllib.request
import os
import pyreaddbc
from dbfread import DBF
import pandas as pd
import gc

def ftp_extract_from_pysus(year: int, month: int):
    # 1. Parâmetros
    v_estado = "GO"
    v_ano = year
    v_mes = month
    v_grupo = "PA" # Produção Ambulatorial

    ano_curto = str(v_ano)[2:]
    mes_str = str(v_mes).zfill(2)
    nome_arquivo = f"{v_grupo}{v_estado}{ano_curto}{mes_str}.dbc"

    # 2. Caminhos
    url_ftp = f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIASUS/200801_/Dados/{nome_arquivo}"
    caminho_temp_dbc = f"/tmp/{nome_arquivo}"
    caminho_temp_dbf = caminho_temp_dbc.replace(".dbc", ".dbf")

    # Caminho final padronizado da Landing Zone (Será criado como diretório para os lotes)
    caminho_landing = f"/Volumes/oftalmo_sus/00_landing/raw_files/SIA/SIA_{v_grupo}_{v_estado}_{v_ano}{mes_str}.parquet"

    print(f"📥 Acessando FTP do governo (Arquitetura Anti-Crash / Chunking)...")
    print(f"🔗 URL: {url_ftp}")

    try:
        # Passo A: Download do arquivo bruto (.dbc)
        print("⏳ Baixando o arquivo do DATASUS...")
        urllib.request.urlretrieve(url_ftp, caminho_temp_dbc)
        print("✅ Download concluído!")
        
        # Passo B: Descompactação (.dbc para .dbf)
        print("🔄 Descompactando a criptografia do governo...")
        pyreaddbc.dbc2dbf(caminho_temp_dbc, caminho_temp_dbf)
        
        # Passo C: Leitura do banco em lotes iteráveis (Proteção de RAM)
        print("💾 Iniciando processamento em Lotes para proteger o Cluster...")
        tabela_dbf = DBF(caminho_temp_dbf, encoding='iso-8859-1')
        
        # Cria o diretório pai que armazenará os pequenos arquivos Parquet
        os.makedirs(caminho_landing, exist_ok=True)
        
        registros = []
        tamanho_lote = 250000 # Descarrega a memória a cada 250 mil linhas
        lote_id = 1
        total_linhas = 0
        
        for record in tabela_dbf:
            registros.append(record)
            
            # Atingiu o limite seguro? Salva e limpa a memória
            if len(registros) >= tamanho_lote:
                df_chunk = pd.DataFrame(registros)
                
                # Salva o lote dentro do diretório caminho_landing
                caminho_arquivo_lote = f"{caminho_landing}/parte_{lote_id}.parquet"
                df_chunk.to_parquet(caminho_arquivo_lote, index=False)
                
                total_linhas += len(registros)
                print(f"✅ Lote {lote_id} salvo! ({total_linhas:,} linhas processadas...)")
                
                # Limpeza explícita da RAM
                registros.clear()
                del df_chunk
                gc.collect()
                lote_id += 1
                
        # Passo D: Salva o saldo final (linhas que não completaram um lote cheio)
        if registros:
            df_chunk = pd.DataFrame(registros)
            caminho_arquivo_lote = f"{caminho_landing}/parte_{lote_id}.parquet"
            df_chunk.to_parquet(caminho_arquivo_lote, index=False)
            total_linhas += len(registros)
            print(f"✅ Lote final salvo! Totalizando {total_linhas:,} linhas.")

        print(f"🚀 Arquivos salvos em: {caminho_landing}")
        
        # Retorna o path do diretório particionado para uso na próxima etapa
        return caminho_landing

    except Exception as e:
        import traceback
        print(f"❌ Erro na extração direta: {e}")
        traceback.print_exc()
        return None

In [0]:
from pyspark.sql.functions import col, count, desc

def valida_periodo_ftp_pysus(caminho_landing:str, month:int, year:int):
    """
    Verifica se o período extraído do FTP do DATASUS é válido comparando com o sigtap silver.
    """

    # 1. Caminhos - Ajuste caso o nome do ficheiro difira ligeiramente
    #caminho_landing_sia = "/Volumes/oftalmo_sus/00_landing/raw_files/sia_pa/SIA_PA_GO_202410.parquet"
    caminho_landing_sia = caminho_landing
    caminho_silver_sigtap = "oftalmo_sus.02_silver.tb_dim_sigtap_oftalmo" 

    print(f"🚀 A verificar os dados massivos da Landing Zone ({month}/{year})...")

    try:
        # 2. Leitura dos Ficheiros
        df_sia_landing = spark.read.format("parquet").load(caminho_landing_sia)
        df_sigtap_oftalmo = spark.read.table(caminho_silver_sigtap)
        
        total_sia = df_sia_landing.count()
        print(f"✅ Total REAL de registos no SIA (GO - {month}/{year}): {total_sia:,}")
        
        if total_sia > 35000:
            print("🎉 SUCESSO! Ultrapassámos a barreira da amostra truncada!")
        
        # 3. Preparação para o Cruzamento: Igualar Tipos e Formatos
        # A base do FTP (.dbc) geralmente traz o PA_PROC_ID sem zeros à esquerda se convertido diretamente para numérico pelo Pandas
        print("\n🔄 A preparar chaves para o cruzamento oftalmológico...")
        df_sia_clean = df_sia_landing.withColumn("PA_PROC_ID_NUM", col("PA_PROC_ID").cast("bigint"))
        df_sigtap_clean = df_sigtap_oftalmo.withColumn("CO_PROC_NUM", col("CO_PROCEDIMENTO").cast("bigint"))
        
        # 4. A Hora da Verdade: INNER JOIN com o seu SIGTAP Oftalmológico
        print("👁️ A cruzar dados reais com o catálogo de Oftalmologia...")
        df_oftalmo_real = df_sia_clean.join(
            df_sigtap_clean,
            df_sia_clean["PA_PROC_ID_NUM"] == df_sigtap_clean["CO_PROC_NUM"],
            "inner"
        )
        
        total_oftalmo = df_oftalmo_real.count()
        print(f"\n🎯 Total de procedimentos de OFTALMOLOGIA encontrados: {total_oftalmo:,}")
        
        if total_oftalmo > 0:
            print("\n🏆 Top 5 Procedimentos Oftalmológicos Mais Realizados no Mês:")
            
            # Agrupamento simples para demonstrar o valor da análise
            resumo_procedimentos = df_oftalmo_real.groupBy("NO_PROCEDIMENTO") \
                                                .agg(count("*").alias("Total_Realizado")) \
                                                .orderBy(desc("Total_Realizado"))
            display(resumo_procedimentos.limit(5))
            
        else:
            print("❌ Alerta: A base gigante carregou, mas a oftalmologia ainda retornou zero. O Join pode precisar de um ajuste de zeros à esquerda novamente.")

    except Exception as e:
        print(f"❌ Erro na leitura ou processamento: {e}")

In [0]:
dbutils.widgets.text("year", "2025")
dbutils.widgets.text("month", "06")

try:
    ano = dbutils.widgets.get("year")
    mes = dbutils.widgets.get("month")
except ValueError:
    print("❌ Erro:")

print(f"Periodo: {ano}{mes}")

caminho_landing = ftp_extract_from_pysus(ano, mes)

if caminho_landing is not None:
    valida_periodo_ftp_pysus(caminho_landing, mes, ano)
else:
    print(f"⚠️ Pulando validação para {mes}/{ano} - arquivo não disponível no FTP")